In [0]:
path = "abfss://landing-zone@lshc.dfs.core.windows.net/hospital/"
display(dbutils.fs.ls(path))

In [0]:
path = "abfss://landing-zone@lshc.dfs.core.windows.net/edi_837/"
display(dbutils.fs.ls(path))

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

STORAGE_ACCOUNT = "lshc"
CONTAINER_NAME = "landing-zone"
BASE_ADLS = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

SOURCE_CONFIG = {
    "hospital": {
        "source_subpath": "hospital/",
        "format": "parquet",
        "options": {},
        "target_table_path": "delta/bronze_hospital_claims",
        "checkpoint_subpath": "checkpoints/bronze_hospital",
        "schema_subpath": "schema/bronze_hospital",
    },
    "edi_837": {
        "source_subpath": "edi_837/",
        "format": "json",
        "options": {"multiline": "false"},  # Set to false for newline-delimited JSON (NDJSON)
        "target_table_path": "delta/bronze_edi_837_claims",
        "checkpoint_subpath": "checkpoints/bronze_edi_837",
        "schema_subpath": "schema/bronze_edi_837",
    }
}

def ingest_bronze_feed(feed_name: str, config: dict):
    source_path = f"{BASE_ADLS}/{config['source_subpath']}"
    target_path = f"{BASE_ADLS}/{config['target_table_path']}"
    checkpoint_path = f"{BASE_ADLS}/{config['checkpoint_subpath']}"
    schema_path = f"{BASE_ADLS}/{config['schema_subpath']}"

    print(f"==================================================")
    print(f"Processing Feed: [{feed_name}]")
    print(f"Source:     {source_path}")
    print(f"Target:     {target_path}")
    print(f"==================================================")

    # Auto Loader read stream
    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", config["format"])
        .option("cloudFiles.schemaLocation", schema_path)
    )

    for k, v in config["options"].items():
        reader = reader.option(k, v)

    stream_df = (
        reader.load(source_path)
        .withColumn("_source_feed", lit(feed_name))
        .withColumn("_ingested_file_name", input_file_name())
        .withColumn("_ingested_at", current_timestamp())
    )

    # Append to Delta Lake with checkpointing
    query = (
        stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(target_path)
    )

    query.awaitTermination()
    print(f"Feed [{feed_name}] successfully ingested.\n")

In [0]:
# Ingest Hospital and EDI-837 sequentially
for feed_name, cfg in SOURCE_CONFIG.items():
    ingest_bronze_feed(feed_name, cfg)

print("All pending Bronze feeds have been processed.")

In [0]:
# Hospital Claims
hosp_path = f"{BASE_ADLS}/delta/bronze_hospital_claims"
df_hosp = spark.read.format("delta").load(hosp_path)
df_hosp.createOrReplaceTempView("bronze_hospital_claims")

# EDI-837 Claims
edi_path = f"{BASE_ADLS}/delta/bronze_edi_837_claims"
df_edi = spark.read.format("delta").load(edi_path)
df_edi.createOrReplaceTempView("bronze_edi_837_claims")

print(f"Hospital records count: {df_hosp.count()}")
print(f"EDI-837 records count:  {df_edi.count()}")

In [0]:
%sql
SELECT distinct hospital_claim_id
FROM bronze_hospital_claims


In [0]:
%sql
SELECT 
    -- Method A: Using Databricks JSON path operator (:)
    claim_information:claim_id::string AS claim_id,
    subscriber:member_id::string AS member_id,
    billing_provider:organization_name::string AS provider,
    CAST(claim_information:total_claim_charge_amount AS DOUBLE) AS total_amount,
    claim_information:diagnosis_codes AS diagnoses,
    claim_information:service_lines AS service_lines,
    _ingested_at
FROM bronze_edi_837_claims


In [0]:
# 1. Check what physical files actually exist in ADLS Landing Zone
print("=== FILES IN ADLS HOSPITAL FOLDER ===")
try:
    hosp_files = dbutils.fs.ls(f"{BASE_ADLS}/hospital/")
    for f in hosp_files:
        print(f.name, f.size)
except Exception as e:
    print(f"Error checking hospital files: {e}")

print("\n=== FILES IN ADLS EDI_837 FOLDER ===")
try:
    edi_files = dbutils.fs.ls(f"{BASE_ADLS}/edi_837/")
    for f in edi_files:
        print(f.name, f.size)
except Exception as e:
    print(f"Error checking EDI files: {e}")

# 2. Check Delta Lake Commit History
print("\n=== DELTA COMMIT HISTORY (HOSPITAL) ===")
display(spark.sql(f"DESCRIBE HISTORY delta.`{BASE_ADLS}/delta/bronze_hospital_claims`"))

In [0]:
BASE_ADLS = "abfss://landing-zone@lshc.dfs.core.windows.net"

# We must delete the Delta tables, Auto Loader checkpoints, and inferred schemas
directories_to_wipe = [
    f"{BASE_ADLS}/delta/bronze_clinic_claims",
    f"{BASE_ADLS}/delta/bronze_hospital_claims",
    f"{BASE_ADLS}/delta/bronze_edi_837_claims",
    f"{BASE_ADLS}/checkpoints",
    f"{BASE_ADLS}/schema"
]

for d in directories_to_wipe:
    try:
        # recurse=True drops the folder and everything inside it
        dbutils.fs.rm(d, recurse=True)
        print(f"Successfully deleted: {d}")
    except Exception as e:
        print(f"Directory not found or already deleted: {d}")

print("\nEnvironment is clean! You are ready to run ingest_bronze.py.")